# Data lifecycle (clean old assets)
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-156

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

In [ ]:
# Other imports
import json
import math

In [ ]:
col_names = []
original_items = []

# Create n test collections
for i in range(2):
    col_name = f"collection_lifecycle_{i}"
    print(f"Stage items in {col_name!r}")
    col_names.append(col_name)
    create_test_collection(col_name)

    # Stage auxip and cadip products
    nb_of_objects = 10 # n objects max
    original_items.extend(stage_test_objects(auxip_client, nb_of_objects, col_name))
    original_items.extend(stage_test_objects(cadip_client, nb_of_objects, col_name))
    
original_items.sort(key=lambda item: item.id)

In [ ]:
def print_items(items: list[Item]):
    print("Staged items:")
    for item in items:
        info = {
            "id": item.id,
            "properties": {
                "expires": item.properties["expires"],
                "unpublished": item.properties.get("unpublished", None),
            },
            "assets": f"# {len(item.assets)} asset{'s' if len(item.assets) > 1 else ''}",
        }
        print(json.dumps(info, indent=2))
print_items(original_items)

In [ ]:
# In local mode, call manually the catalog endpoint to trigger the data lifecycle management.
# In cluster mode, you have to wait for the next task to be triggered automatically.
def trigger_lifecycle():
    if local_mode:
        http_session.get(f"{catalog_client.href_service}/data/lifecycle").raise_for_status()		
    else:
        print("""Wait for next cleaning. Go to Grafana then:
  -> Explore 
  -> Loki 
  -> app = rs-server-catalog 
  -> Line contains = data_lifecycle
""")
trigger_lifecycle()

In [ ]:
# Compare two list of items
def compare(l1: list[Item], l2: list[Item]):
    len1 = len(l1)
    len2 = len(l2)
    assert len1 == len2, f"Lists have different lengths: {len1} vs {len2}"
    for i in range(len1):
        d1 = l1[i].to_dict()
        d2 = l2[i].to_dict()        
        assert d1 == d2, f"Different values for item #{i}:\n{json.dumps(d1, indent=2)}\n{json.dumps(d2, indent=2)}"

In [ ]:
# For now, the items have not changed after triggering the lifecyle
# and GETting them from the catalog, because they are not expired.
unchanged_items = catalog_client.search(
    owner_id=OWNER_ID,
    collections=col_names,
    max_items=10e3,
    sortby="id",
).items
compare(original_items, unchanged_items)

In [ ]:
a = unchanged_items.items[0].to_dict()
b = items[0].to_dict()
unchanged_items.items == items
# Compa